# Notebook 05: Modelling Data Preparation

## Objective

This notebook prepares separate datasets for the econometric and machine learning stages of the research.

The econometric dataset will support the analysis of symmetric and asymmetric exchange-rate pass-through. The machine learning dataset will support time-aware model training and evaluation.

This notebook focuses on modelling decisions and does not repeat the exploratory analysis completed in Notebook 04.

In [11]:
# Import Libraries
from pathlib import Path
import pandas as pd

The feature-engineered dataset produced in Notebook 03 is used as the input for modelling preparation. Notebook 04 is not used as a data source because it contains exploratory transformations created for analysis rather than for the final modelling pipeline.

In [12]:
# create file path
DATA_DIR = Path("../data/processed")
featured_df = DATA_DIR / "featured_data.csv"

In [13]:
data = pd.read_csv(
    featured_df, 
    parse_dates=["Date"]
    )

data = data.sort_values(
    ["Eight digit code", "Date"]
).reset_index(drop=True)

In [14]:
# preview the dataset
data.head()

,Date,DivisionDescription,GroupDescription,ClassDescription,SubclassDescription,Weight,Eight digit code,CPI,ExchangeRate,Year,...,ExchangeRate_Lag1,ExchangeRate_Lag3,ExchangeRate_Lag6,CPI_Lag1,CPI_Lag3,ExchangeRate_Change,Depreciation,Appreciation,ExchangeRate_MA3,ExchangeRate_STD3
0,2015-07-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,57.2,12.452687,2015,...,12.303048,12.012553,11.566500,57.0,57.0,0.012163,0.012163,0.000000,12.241892,0.247116
1,2015-08-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,59.3,12.914595,2015,...,12.452687,11.969940,11.577440,57.2,57.3,0.037093,0.037093,0.000000,12.556777,0.318784
2,2015-09-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,58.9,13.610852,2015,...,12.914595,12.303048,12.068727,59.3,57.0,0.053912,0.053912,0.000000,12.992711,0.583021
3,2015-10-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,59.5,13.504323,2015,...,13.610852,12.452687,12.012553,58.9,57.2,-0.007827,0.000000,0.007827,13.343257,0.375034
4,2015-11-01,Food and non-alcoholic beverages,Food,Cereal products,Cereals,0.540243,1111201,58.9,14.125733,2015,...,13.504323,12.914595,11.969940,59.5,59.3,0.046016,0.046016,0.000000,13.746969,0.332316


In [15]:
# confirm the dataset shape
print("Dataset shape:", data.shape)

Dataset shape: (14150, 22)


Each observation should represent one eight-digit food item in one month.

The eight-digit code is used as the item identifier because multiple food items may share the same subclass description. Before preparing either modelling dataset, the combination of eight-digit code and date must therefore be unique.

In [16]:
# check the item-month structure
modelling_key = ["Eight digit code", "Date"]

duplicate_count = data.duplicated(subset=modelling_key).sum()

print(
    "Unique eight-digit codes:",
    data["Eight digit code"].nunique()
)

print(
    "Unique subclass descriptions:",
    data["SubclassDescription"].nunique()
)

print(
    "Duplicate item-month observations:",
    duplicate_count
)

Unique eight-digit codes: 134
Unique subclass descriptions: 49
Duplicate item-month observations: 0


### Interpretation of the Modelling Unit

The dataset contains 134 unique eight-digit food item codes across 49 subclass descriptions. This confirms that multiple individual food items may belong to the same subclass.

The combination of `Eight digit code` and `Date` is therefore used as the modelling key. This preserves the individual CPI series within each food subclass.

No duplicate item-month observations were identified, confirming that each row represents a unique food item observed in a specific month.

In [17]:
# Summarize the coverage of each food item
item_coverage = (
    data.groupby("Eight digit code")
    .agg(
        start_date=("Date", "min"),
        end_date=("Date", "max"),
        observations=("Date", "count")
    )
    .reset_index()
)

year_difference = (
    item_coverage["end_date"].dt.year - item_coverage["start_date"].dt.year
)

month_difference = (
    item_coverage["end_date"].dt.month - item_coverage["start_date"].dt.month
)

item_coverage["expected_months"] = (year_difference * 12 + month_difference + 1 )

item_coverage["missing_months"] = (
    item_coverage["expected_months"] - item_coverage["observations"]
)

In [18]:
# Review the panel structure
print("Overall start date:", data["Date"].min().date())
print("Overall end date:", data["Date"].max().date())
print("Unique months:", data["Date"].nunique())

print(
    "Items with missing months:",
    (item_coverage["missing_months"] > 0).sum()
)

item_coverage["observations"].describe()

Overall start date: 2015-07-01
Overall end date: 2025-12-01
Unique months: 126
Items with missing months: 0


count    134.000000
mean     105.597015
std       40.044359
min       10.000000
25%      105.000000
50%      126.000000
75%      126.000000
max      126.000000
Name: observations, dtype: float64

In [19]:
# view items with the least observations
item_coverage.sort_values(
    ["observations", "Eight digit code"]
).head(10)

,Eight digit code,start_date,end_date,observations,expected_months,missing_months
1,1111202,2025-03-01,2025-12-01,10,10,0
4,1112301,2025-03-01,2025-12-01,10,10,0
32,1124002,2025-03-01,2025-12-01,10,10,0
33,1124003,2025-03-01,2025-12-01,10,10,0
38,1125104,2025-03-01,2025-12-01,10,10,0
39,1125105,2025-03-01,2025-12-01,10,10,0
41,1125901,2025-03-01,2025-12-01,10,10,0
42,1125902,2025-03-01,2025-12-01,10,10,0
47,1133902,2025-03-01,2025-12-01,10,10,0
48,1134101,2025-03-01,2025-12-01,10,10,0


### Interpretation of the Panel Structure

The dataset covers 126 months from July 2015 to December 2025 and contains no internal gaps within the individual food-item series.

However, the number of observations per item ranges from 10 to 126. The panel is therefore unbalanced because some food items enter or leave the dataset at different points during the study period.

The shorter series should not be treated as missing data because their observations are continuous within their available periods. No additional months will be imputed.

In [20]:
# summarise item coverage patterns
coverage_patterns = (
    item_coverage.groupby(
        ["start_date", "end_date", "observations"]
    )
    .size()
    .reset_index(name="item_count")
    .sort_values(
        ["start_date", "end_date", "observations"]
    )
)

coverage_patterns

,start_date,end_date,observations,item_count
0,2015-07-01,2025-12-01,126,95
1,2017-04-01,2025-12-01,105,17
2,2022-04-01,2025-12-01,45,5
3,2025-03-01,2025-12-01,10,17


In [22]:
# Count items covering the full modelling period
overall_start_date = data["Date"].min()
overall_end_date = data["Date"].max()

full_period_items = (
    item_coverage["start_date"].eq(overall_start_date)
    & item_coverage["end_date"].eq(overall_end_date)
).sum()

print("Items covering the full period:", full_period_items)
print(
    "Items with fewer than 60 observations:",
    (item_coverage["observations"] < 60).sum()
)
print(
    "Items introduced during 2025:",
    (item_coverage["start_date"].dt.year == 2025).sum()
)

Items covering the full period: 95
Items with fewer than 60 observations: 22
Items introduced during 2025: 17


### Interpretation of Item Coverage

The panel contains four groups of food items based on their starting dates. Ninety-five items cover the full 126 month modelling period, while the remaining items entered the dataset in 2017, 2022 or 2025.

All item series continue until December 2025 and contain no internal missing months. The unequal series lengths therefore reflect the later introduction of certain food items rather than incomplete observations within their available periods.

The 2022 and 2025 series may be too short for reliable item-level econometric modelling. Their food-category coverage must be examined before deciding whether they should be excluded.